In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import copy
from datetime import datetime, timedelta

from sklearn.metrics import balanced_accuracy_score
import optuna
from optuna.samplers import TPESampler
from sklearn.utils.class_weight import compute_class_weight
import sys
import os
import joblib
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import glob

sys.path.append("/glade/u/home/jhayron/WR_Predictability/3_MLModels/")
# from model_builders_v2 import *

/glade/work/jhayron/conda-envs/cnn_wr/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from sklearn.decomposition import PCA
from scipy.stats import chi2_contingency

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.path as mpath
import cartopy
import cartopy.crs as ccrs
import cartopy.feature
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
import cartopy.feature as cf
import shapely.geometry as sgeom

In [3]:
wr_series = pd.read_csv('/glade/work/jhayron/Data4Predictability/WR_Series_20231120.csv',\
                index_col=0,names=['week0'],skiprows=1,parse_dates=True)
for wk in range(2,10):
    series_temp = copy.deepcopy(wr_series["week0"])
    series_temp.index = series_temp.index - timedelta(weeks = wk-1)
    series_temp.name = f'week{wk-1}'
    if wk==2:
        df_shifts = pd.concat([pd.DataFrame(wr_series["week0"]),pd.DataFrame(series_temp)],axis=1)  
    else:
        df_shifts = pd.concat([df_shifts,pd.DataFrame(series_temp)],axis=1)

In [4]:
path_weekly_anoms = '/glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/'

In [5]:
files_weekly_anoms = np.sort(glob.glob(f'{path_weekly_anoms}/*.nc'))

# Iterate through all variables

In [6]:
names_wr = ['WR1: AR','WR2: PT', 'WR3: GH', 'WR4: CUSH', 'No WR']

In [7]:
try: os.mkdir(f'/glade/u/home/jhayron/WR_Predictability/6_PCA_xgboost/figures_detrended_20240317/{name_var}/')
except: pass

try: os.mkdir(path_figures)
except: pass

In [8]:
for ifile in range(len(files_weekly_anoms)):
    print(ifile,files_weekly_anoms[ifile])

0 /glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/IC_SODA.nc
1 /glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/IT_SODA.nc
2 /glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/MLD_SODA.nc
3 /glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/OHC100_SODA.nc
4 /glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/OHC200_SODA.nc
5 /glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/OHC300_SODA.nc
6 /glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/OHC50_SODA.nc
7 /glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/OHC700_SODA.nc
8 /glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/OLR_ERA5.nc
9 /glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/SD_ERA5.nc
10 /glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/SSH_SODA.n

In [9]:
def detect_outliers_zscore(data, threshold=4):
    """
    Detect outliers in a univariate dataset using the Z-score method.

    Parameters:
    - data: A 1D numpy array representing the time series data.
    - threshold: The threshold value for the Z-score. Data points with a Z-score greater than this value are considered outliers.

    Returns:
    - An array of indices corresponding to the outliers.
    """
    z_scores = (data - np.mean(data)) / np.std(data)
    outlier_indices = np.where(np.abs(z_scores) > threshold)[0]
    return outlier_indices

In [11]:
for ifile in range(len(files_weekly_anoms)):
# for ifile in range(13, 22):
# for ifile in [10,20,21]:
# for ifile in [8]:
    file_temp = files_weekly_anoms[ifile]
    dataset = xr.open_dataset(file_temp)
    var_name = list(dataset.data_vars.keys())[0]
    print(file_temp,var_name)
    dataset = dataset.load()

    name_var = file_temp.split('/')[-1][:-3]
    flattened_da = dataset.stack(flat=('lat','lon')).transpose('time','flat')[var_name]
    flattened_da_clean = copy.deepcopy(flattened_da)
    
    nan_mask = flattened_da.isnull().any(dim='time')
    where_nonan = np.where(nan_mask==False)[0]

    if ('SSH' in file_temp)|\
        ('OHC' in file_temp)|\
        ('Z500' in file_temp)|\
        ('OLR' in file_temp)|\
        ('SST' in file_temp)|\
        ('U10' in file_temp)|\
        ('U200' in file_temp):
        pass
    else:
        for icolumn in where_nonan:
            series_temp = pd.Series(flattened_da_clean[:,icolumn],index=flattened_da_clean[:,icolumn].time)
            series_temp = series_temp[series_temp.index.dayofweek == 0]
        
            rolling_mean = series_temp.rolling(timedelta(days=180),center=True).mean()
            # rolling_std = rolling_mean.rolling(timedelta(days=30),center=True).std()
            # outliers  = detect_outliers_zscore(rolling_std)
            diff = rolling_mean.diff()
            outliers  = detect_outliers_zscore(diff)
            if len(outliers)>=5:
                flattened_da_clean[:, icolumn] = np.nan
            
    clean_dataset = flattened_da_clean.unstack('flat')
    flattened_da = clean_dataset.stack(flat=('lat','lon')).transpose('time','flat')#[var_name]
    # nan_mask = flattened_da.isnull().all(dim='time')
    nan_mask = flattened_da.isnull().any(dim='time')
    where_nonan = np.where(nan_mask==False)[0]
    flattened_nonan = flattened_da[:,where_nonan]
    flattened_nonan = flattened_nonan.where(flattened_nonan >= -10, 0)
    flattened_nonan = flattened_nonan.where(flattened_nonan <= 10, 0)
    if 'OLR' in file_temp:
        flattened_nonan = flattened_nonan*(-1)
    # aaaa
    # create pca object
    pca_obj = PCA(20, whiten=True)##
    pca_obj = pca_obj.fit(flattened_nonan)

    # transform era5 data with pca
    transformed_da = pca_obj.transform(flattened_nonan)

    print(f'number of pcs: {len(pca_obj.explained_variance_ratio_)}')
    transformed_da = pd.DataFrame(transformed_da,index = dataset.time)


    df_wr_temp = df_shifts['week0'].dropna()
    mean_hist_wrs = np.histogram(df_wr_temp,bins=np.arange(-0.5,5,1))[0]
    mean_hist_wrs = mean_hist_wrs/np.sum(mean_hist_wrs)

    try: os.mkdir(f'/glade/u/home/jhayron/WR_Predictability/6_PCA_xgboost/figures_detrended_20240317/{name_var}/')
    except: pass

    transformed_da.to_csv(f'/glade/u/home/jhayron/WR_Predictability/6_PCA_xgboost/figures_detrended_20240317/{name_var}/PC_{name_var}.csv')
    
    try: os.mkdir(f'/glade/u/home/jhayron/WR_Predictability/6_PCA_xgboost/data_pcs/Heatmaps/{name_var}/')
    except: pass
    try: os.mkdir(f'/glade/u/home/jhayron/WR_Predictability/6_PCA_xgboost/data_pcs/TimeSeries/{name_var}/')
    except: pass
    try: os.mkdir(f'/glade/u/home/jhayron/WR_Predictability/6_PCA_xgboost/data_pcs/EOFs/{name_var}/')
    except: pass
    transformed_da.to_csv(f'/glade/u/home/jhayron/WR_Predictability/6_PCA_xgboost/data_pcs/TimeSeries/{name_var}/PC_{name_var}.csv')            
    for ipc in range(len(pca_obj.explained_variance_ratio_)):
        # print(f'PC: {ipc}**********************')
        ##########Unstack PC #####################
        pc_flattened_nonan = copy.deepcopy(flattened_nonan.isel(time=0))#.values
        # flattened_nonan.isel(time=0).values = pca_obj.components_[0]
        pc_flattened_nonan.values = pca_obj.components_[ipc]
        pc_flattened = copy.deepcopy(flattened_da.isel(time=0))
        array_temp = copy.deepcopy(pc_flattened.values)
        array_temp[where_nonan] = pc_flattened_nonan.values
        pc_flattened.values  = array_temp
        unstacked_pc = pc_flattened.unstack('flat')
        unstacked_pc.to_netcdf(f'/glade/u/home/jhayron/WR_Predictability/6_PCA_xgboost/data_pcs/EOFs/{name_var}/EOF_{ipc}.nc')
        ##########PLOT MAP #####################
        quintiles = np.percentile(transformed_da[ipc], [20, 40, 60, 80])
        median_series = np.median(transformed_da[ipc])
        
        ##########Normalize PC #####################
        normalized_pc = transformed_da[ipc]#/np.nanmax(abs(transformed_da[ipc]))
        normalized_pc.index = normalized_pc.index.floor('D')

        ##########Get conditional histogram for each bin within PC #####################
        bins = np.percentile(normalized_pc,[0,20,40,60,80,100])
        np.save(f'/glade/u/home/jhayron/WR_Predictability/6_PCA_xgboost/data_pcs/TimeSeries/{name_var}/bins_{ipc}.npy',bins)
        mid_bins = np.array([10,30,50,70,90])

        avg_change_bins_weeks = np.zeros([len(mid_bins),len([1,2,3,4,5,6,7,8])])
        p_val_bins_weeks = np.zeros([len(mid_bins),len([1,2,3,4,5,6,7,8])])
        
        for week in [1,2,3,4,5,6,7,8]:
            df_pc_wr = pd.concat([normalized_pc,df_shifts[f'week{week}']],axis=1).dropna()
            p_values = []
            prob2d = np.zeros([5,len(mid_bins)])
            for ibin in range(len(bins)-1):
                # print(bin_center)
                df_temp = df_pc_wr[(df_pc_wr[ipc]>=bins[ibin])&(df_pc_wr[ipc]<=bins[ibin+1])][f'week{week}']
                hist_temp = np.histogram(df_temp,bins=np.arange(-0.5,5,1))[0]
                hist_temp = hist_temp/np.sum(hist_temp)
                diff_temp = hist_temp - mean_hist_wrs
                prob2d[:,ibin] = diff_temp
                
                freq_table_A = df_wr_temp.value_counts().sort_index()##
                freq_table_B = df_temp.value_counts().sort_index()##
                observed = pd.concat([freq_table_A, freq_table_B], axis=1).fillna(0).values##
                chi2, p, _, _ = chi2_contingency(observed)##
                p_values.append(p)##
            p_values = np.array(p_values)##

            avg_hist_change_for_week = np.mean(abs(prob2d),axis=0)
            p_val_bins_weeks[:,week-1] = p_values
            avg_change_bins_weeks[:,week-1] = avg_hist_change_for_week
            np.save(f'/glade/u/home/jhayron/WR_Predictability/6_PCA_xgboost/data_pcs/Heatmaps/{name_var}/prob2d_{ipc}_{week}.npy',prob2d)
        
        np.save(f'/glade/u/home/jhayron/WR_Predictability/6_PCA_xgboost/data_pcs/Heatmaps/{name_var}/avg_change_bins_weeks_{ipc}.npy',avg_change_bins_weeks)
        np.save(f'/glade/u/home/jhayron/WR_Predictability/6_PCA_xgboost/data_pcs/Heatmaps/{name_var}/p_val_bins_weeks_{ipc}.npy',p_val_bins_weeks)


/glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/IC_SODA.nc cn_total_anomalies
number of pcs: 20
/glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/IT_SODA.nc hi_anomalies
number of pcs: 20
/glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/MLD_SODA.nc mlt_anomalies
number of pcs: 20
/glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/OHC100_SODA.nc ocean_heat_content_anomalies
number of pcs: 20
/glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/OHC200_SODA.nc ocean_heat_content_anomalies
number of pcs: 20
/glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/OHC300_SODA.nc ocean_heat_content_anomalies
number of pcs: 20
/glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/OHC50_SODA.nc ocean_heat_content_anomalies
number of pcs: 20
/glade/derecho/scratch/jhayron/Data4Predictability/WeeklyAnoms_DetrendedStd/OHC700_SODA

In [ ]:
transformed_da